# DEX — yolo11s fine-tune on workshop footage (Kaggle GPU)

**Maqsad:** apne workshop ke clips pe yolo11s ko fine-tune karna taake mushkil cases
(jhuka hua mechanic ~0.05 conf, raat ka worker ~0.43) behtar detect hon — laptop CPU
pe training namumkin hai, Kaggle ka free GPU (T4/P100, ~30 hr/week) use hota hai.

**Pipeline:** clips se frames → bara model (yolo11x) GPU pe auto-label karta hai →
person-only YOLO dataset → yolo11s fine-tune → pehle/baad ka muqabla → `best.pt` download.

**Chalane ka tareeqa:**
1. Factory pe `tools\record_clips.bat --seconds 300` chalao (kaam hote waqt, din + raat dono)
2. `clips/` folder ko Kaggle Dataset bana ke upload karo (naam: `dex-workshop-clips`)
3. Yeh notebook Kaggle pe import karo, right panel: **Accelerator = GPU**, dataset attach karo
4. Run All → aakhri cell `best.pt` deta hai → laptop pe `models/` mein rakho aur
   `config.yaml` mein `model: models/yolo11s_dex.pt` kar do

**Khabardaar:** auto-label sirf utna acha hai jitna bara model. Cell 5 ka review grid
zaroor dekho — ghalat labels dikhen to `LABEL_CONF` barhao.

In [ ]:
%pip -q install ultralytics
import torch
from pathlib import Path

CLIPS_DIR = Path('/kaggle/input/dex-workshop-clips')   # attached dataset
WORK = Path('/kaggle/working')
assert torch.cuda.is_available(), 'GPU on nahi hai — right panel se Accelerator = GPU karo'
clips = sorted(CLIPS_DIR.rglob('*.mp4'))
print(f'GPU: {torch.cuda.get_device_name(0)} | clips: {len(clips)}')
assert clips, 'dex-workshop-clips dataset attach karo (record_clips.bat ke mp4 files)'

## 1. Frames nikaalo (har clip se, 1 frame/sec)

In [ ]:
import cv2

FRAMES = WORK / 'frames'
FRAMES.mkdir(exist_ok=True)
EVERY_S = 1.0                      # 1 frame/sec — 5 min clip = ~300 frames

n = 0
for clip in clips:
    cap = cv2.VideoCapture(str(clip))
    fps = cap.get(cv2.CAP_PROP_FPS) or 12
    step = max(1, int(round(fps * EVERY_S)))
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % step == 0:
            cv2.imwrite(str(FRAMES / f'{clip.stem}_{i:06d}.jpg'), frame)
            n += 1
        i += 1
    cap.release()
print(f'{n} frames -> {FRAMES}')

## 2. Auto-label: yolo11x (GPU) har frame pe person boxes banata hai
Sirf confident labels rakhte hain — kamzor detections training ko kharab karti hain.

In [ ]:
from ultralytics import YOLO

LABEL_CONF = 0.45     # auto-label ka minimum conf — review grid dekh kar adjust karo
IMG_TRAIN = 512       # wahi imgsz jo production pipeline use karti hai

labeler = YOLO('yolo11x.pt')
frames = sorted(FRAMES.glob('*.jpg'))
labels = {}           # frame path -> list[(cx, cy, bw, bh)] normalized
for batch_start in range(0, len(frames), 64):
    batch = frames[batch_start:batch_start + 64]
    for r in labeler([str(p) for p in batch], classes=[0], conf=LABEL_CONF,
                     imgsz=960, verbose=False, device=0):
        h, w = r.orig_shape
        boxes = []
        for x1, y1, x2, y2 in r.boxes.xyxy.cpu().numpy():
            boxes.append(((x1 + x2) / 2 / w, (y1 + y2) / 2 / h,
                          (x2 - x1) / w, (y2 - y1) / h))
        labels[Path(r.path)] = boxes

with_person = sum(1 for b in labels.values() if b)
print(f'{with_person}/{len(labels)} frames mein person mila')

## 3. Review grid — labels aankhon se check karo (ghalat = `LABEL_CONF` barhao)

In [ ]:
import random
import matplotlib.pyplot as plt

sample = random.sample([p for p, b in labels.items() if b],
                       k=min(12, with_person))
fig, axes = plt.subplots(3, 4, figsize=(20, 11))
for ax, p in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for cx, cy, bw, bh in labels[p]:
        x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    ax.imshow(img); ax.axis('off'); ax.set_title(p.name[:38], fontsize=8)
for ax in axes.flat[len(sample):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

## 4. YOLO dataset banao (90/10 train/val, clip ke hisaab se split)
Split CLIP level pe hota hai frame level pe nahi — warna val mein train ke
parosi frames aa jate hain aur numbers jhoote acche lagte hain.

In [ ]:
import shutil

DS = WORK / 'dataset'
shutil.rmtree(DS, ignore_errors=True)
for split in ('train', 'val'):
    (DS / 'images' / split).mkdir(parents=True)
    (DS / 'labels' / split).mkdir(parents=True)

stems = sorted({c.stem for c in clips})
val_stems = set(stems[::10]) or {stems[-1]}   # har 10waan clip val mein
kept = {'train': 0, 'val': 0}
for p, boxes in labels.items():
    if not boxes:
        continue                                # background-only frames skip
    split = 'val' if any(p.name.startswith(s) for s in val_stems) else 'train'
    shutil.copy(p, DS / 'images' / split / p.name)
    (DS / 'labels' / split / f'{p.stem}.txt').write_text(
        '\n'.join(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}'
                  for cx, cy, bw, bh in boxes))
    kept[split] += 1

(DS / 'dex.yaml').write_text(
    f'path: {DS}\ntrain: images/train\nval: images/val\n'
    'names:\n  0: person\n')
print(kept)

## 5. Fine-tune yolo11s (backbone pehle 10 layers frozen — thora data, overfit se bachao)

In [ ]:
model = YOLO('yolo11s.pt')
results = model.train(
    data=str(DS / 'dex.yaml'),
    epochs=40, patience=10,
    imgsz=IMG_TRAIN, batch=16, freeze=10,
    lr0=0.002,                 # fine-tune — chhota LR, pretrained na bighre
    degrees=5, scale=0.3, fliplr=0.5, mosaic=0.5,
    project=str(WORK / 'runs'), name='dex_ft', exist_ok=True,
)
BEST = WORK / 'runs' / 'dex_ft' / 'weights' / 'best.pt'
print(BEST, BEST.exists())

## 6. Pehle vs baad — val frames pe stock yolo11s ka muqabla fine-tuned se

In [ ]:
stock = YOLO('yolo11s.pt')
tuned = YOLO(str(BEST))
val_imgs = sorted((DS / 'images' / 'val').glob('*.jpg'))

def mean_top_conf(m):
    confs = []
    for r in m([str(p) for p in val_imgs], classes=[0], conf=0.05,
               imgsz=IMG_TRAIN, verbose=False, device=0):
        c = r.boxes.conf.cpu().numpy()
        confs.append(float(c.max()) if len(c) else 0.0)
    return sum(confs) / max(1, len(confs))

print(f'stock yolo11s  mean top-conf @512: {mean_top_conf(stock):.3f}')
print(f'fine-tuned     mean top-conf @512: {mean_top_conf(tuned):.3f}')
print()
print('Fine-tuned behtar hai to: best.pt download karo (right panel, Output),')
print('laptop pe models/yolo11s_dex.pt rakho, config.yaml -> model: models/yolo11s_dex.pt')
print('PHIR laptop pe apne haard cases (jhuka mechanic, raat) pe detect_probe se verify!')